# Visualisation des reconstructions multi-sampling

Ce notebook charge le pickle produit par `evaluate_multiple_sampling.py`. Pour chaque mélange, il affiche dans une figure unique la mixture, puis la reconstruction moyenne et sa bande $\pm 1\sigma$ pour chaque source et pour le résiduel.

Si les samplings ont été alignés par consensus, l'ordre global des sources reste arbitraire. Pour associer correctement les SNR aux lignes, la moyenne est donc réordonnée une seule fois vers les cibles avec Hungarian. Cette opération ne change ni la moyenne ni son incertitude : elle permute seulement les lignes de la figure.

In [ ]:
from pathlib import Path
import pickle

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from scipy.optimize import linear_sum_assignment

# Chemin relatif au dossier notebooks/. À adapter si nécessaire.
PICKLE_PATH = Path("../results_run_20260828_190432_nb_of_samplings_1000_aligned_to_target_False.pkl")

with PICKLE_PATH.open("rb") as f:
    results = pickle.load(f)

if not results:
    raise RuntimeError("The pickle does not contain any result.")

print(f"Loaded {len(results)} mixtures from {PICKLE_PATH}")

Loaded 9 mixtures from ../results_run_20260828_190432_nb_of_samplings_1000_aligned_to_target_False.pkl


In [ ]:
def to_numpy(value):
    if hasattr(value, "detach"):
        value = value.detach().cpu().numpy()
    return np.asarray(value)


def get_k(result):
    if "K" in result:
        return int(result["K"])

    source_mask = result.get("params", {}).get("source_mask")
    if source_mask is None:
        raise KeyError("K and params['source_mask'] are both missing.")
    return int(to_numpy(source_mask).sum())


def put_mean_in_target_order(prediction_mean, prediction_std, target, k):
    """Apply one global source permutation for labels and SNR display."""
    cost = np.mean(
        (prediction_mean[:k, None] - target[None, :k]) ** 2,
        axis=(-1, -2),
    )
    predicted_slots, target_slots = linear_sum_assignment(cost)

    ordered_mean = prediction_mean.copy()
    ordered_std = prediction_std.copy()
    ordered_mean[target_slots] = prediction_mean[predicted_slots]
    ordered_std[target_slots] = prediction_std[predicted_slots]
    return ordered_mean, ordered_std


CHANNEL_NAMES = ["A real", "A imag.", "E real", "E imag."]
COLORS = {
    "mixture": "#009E73",      # teal green
    "target": "#0072B2",       # deep blue
    "prediction": "#D55E00",   # vermillion
    "uncertainty": "#E69F00",  # amber
}


def style_axis(ax):
    ax.set_facecolor("#FCFCFD")
    ax.grid(color="#CBD5E1", linewidth=0.7, alpha=0.45)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#94A3B8")
    ax.spines["bottom"].set_color("#94A3B8")

In [ ]:
def plot_result(result_position):
    result = results[result_position]
    mixture = to_numpy(result["mixture"])
    target = to_numpy(result["target"])
    prediction_mean = to_numpy(result["prediction_mean"])
    prediction_std = to_numpy(result["prediction_std"])
    k = get_k(result)

    # The consensus has no intrinsic source numbering. This single global
    # permutation associates each plotted row with the correct target/SNR.
    prediction_mean, prediction_std = put_mean_in_target_order(
        prediction_mean, prediction_std, target, k
    )

    params = result.get("params", {})
    snrs = to_numpy(params.get("snr", np.full(k, np.nan))).reshape(-1)

    source_error = np.sum(
        (prediction_mean[:k] - target[:k]) ** 2, axis=(-1, -2)
    )
    source_energy = np.sum(target[:k] ** 2, axis=(-1, -2))
    nmse = source_error / (source_energy + 1e-12)
    nmse_db = 10 * np.log10(nmse + 1e-12)

    uncertainty_rms = np.sqrt(
        np.mean(prediction_std[:k] ** 2, axis=(-1, -2))
    )
    sampling_rms = np.sqrt(
        np.mean(
            prediction_mean[:k] ** 2 + prediction_std[:k] ** 2,
            axis=(-1, -2),
        )
    )
    relative_uncertainty = uncertainty_rms / (sampling_rms + 1e-12)

    n_channels, n_frequency_bins = mixture.shape
    frequency_bins = np.arange(n_frequency_bins)
    n_rows = k + 2  # mixture + K sources + residual
    fig, axes = plt.subplots(
        n_rows,
        n_channels,
        figsize=(4.8 * n_channels, 2.7 * n_rows),
        sharex=True,
        squeeze=False,
    )

    for channel in range(n_channels):
        ax = axes[0, channel]
        ax.plot(
            frequency_bins, mixture[channel],
            color=COLORS["mixture"], lw=1.7, zorder=3
        )
        channel_name = (
            CHANNEL_NAMES[channel]
            if channel < len(CHANNEL_NAMES)
            else f"Channel {channel + 1}"
        )
        ax.set_title(channel_name)
        style_axis(ax)
    axes[0, 0].set_ylabel("Mixture")

    for source in range(k):
        row = source + 1
        snr_text = f"{snrs[source]:.2f}" if source < len(snrs) else "n/a"
        axes[row, 0].set_ylabel(
            f"Source {source + 1}\n"
            f"SNR={snr_text}\n"
            f"NMSE={nmse_db[source]:.2f} dB\n"
            f"σ={uncertainty_rms[source]:.3g}\n"
            f"U={100 * relative_uncertainty[source]:.2f}%"
        )
        for channel in range(n_channels):
            ax = axes[row, channel]
            mean = prediction_mean[source, channel]
            std = prediction_std[source, channel]
            ax.plot(
                frequency_bins, target[source, channel],
                color=COLORS["target"], lw=2.5,
                label="Target", zorder=3
            )
            ax.fill_between(
                frequency_bins, mean - std, mean + std,
                color=COLORS["uncertainty"], alpha=0.18,
                linewidth=0, label="±1 std", zorder=1
            )
            ax.plot(
                frequency_bins, mean, color=COLORS["prediction"],
                lw=1.7, linestyle=(0, (5, 2.5)),
                label="Prediction mean", zorder=4
            )
            style_axis(ax)

    residual_row = k + 1
    axes[residual_row, 0].set_ylabel("Residual / noise")
    for channel in range(n_channels):
        ax = axes[residual_row, channel]
        mean = prediction_mean[-1, channel]
        std = prediction_std[-1, channel]
        ax.plot(
            frequency_bins, target[-1, channel],
            color=COLORS["target"], lw=1.6,
            label="Target", zorder=3
        )
        ax.fill_between(
            frequency_bins, mean - std, mean + std,
            color=COLORS["uncertainty"], alpha=0.18,
            linewidth=0, label="±1 std", zorder=1
        )
        ax.plot(
            frequency_bins, mean, color=COLORS["prediction"],
            lw=1.7, linestyle=(0, (5, 2.5)),
            label="Prediction mean", zorder=4
        )
        style_axis(ax)
        ax.set_xlabel("Frequency bin")

    handles, labels = axes[1, 0].get_legend_handles_labels()
    legend_order = ["Target", "Prediction mean", "±1 std"]
    ordered_handles = [handles[labels.index(label)] for label in legend_order]
    axes[1, 0].legend(
        ordered_handles, legend_order, loc="upper right",
        frameon=True, facecolor="white", edgecolor="#CBD5E1",
        framealpha=0.95,
    )
    matches = result.get("alignment_matches_target", [])
    match_text = (
        f"{sum(matches)}/{len(matches)} match target" if len(matches) else "match unavailable"
    )
    alignment_mode = "target" if result.get("align_to_target", False) else "consensus"
    mixture_index = result.get("mixture_index", result_position)
    fig.suptitle(
        f"Result {result_position} — mixture {mixture_index} — K={k} — "
        f"alignment={alignment_mode} — {match_text} — "
        f"loss={result.get('loss', np.nan):.4g}",
        fontsize=14,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    plt.show()


In [ ]:
result_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(results) - 1,
    step=1,
    description="Result:",
    continuous_update=False,
    layout=widgets.Layout(width="700px"),
)

output = widgets.interactive_output(
    plot_result, {"result_position": result_slider}
)
display(result_slider, output)

IntSlider(value=0, continuous_update=False, description='Result:', layout=Layout(width='700px'), max=8)

Output()